In [1]:
import pandas as pd
import json
import sqlite3

In [2]:
orders = pd.read_csv("orders.csv")
orders.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [4]:
with open("users.json", "r") as f:
    users_data = json.load(f)

users = pd.DataFrame(users_data)
users.head()

,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [5]:
conn = sqlite3.connect("restaurants.db")
cursor = conn.cursor()

In [6]:
with open("restaurants.sql", "r") as f:
    sql_script = f.read()

cursor.executescript(sql_script)
conn.commit()

In [7]:
restaurants = pd.read_sql("SELECT * FROM restaurants", conn)
restaurants.head()

,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [8]:
merged_df = orders.merge(users, on="user_id", how="left")
merged_df = merged_df.merge(restaurants, on="restaurant_id", how="left")

merged_df.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [9]:
merged_df.to_csv("final_food_delivery_dataset.csv", index=False)

In [10]:
merged_df["order_date"] = pd.to_datetime(merged_df["order_date"])
merged_df["quarter"] = merged_df["order_date"].dt.to_period("Q")

C:\Users\user\AppData\Local\Temp\ipykernel_11276\1150937294.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  merged_df["order_date"] = pd.to_datetime(merged_df["order_date"])


In [23]:
gold_city_revenue = merged_df[merged_df["membership"] == "Gold"] \
    .groupby("city")["total_amount"].sum()

gold_city_revenue.sort_values(ascending=False)

city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

In [24]:
merged_df.groupby("cuisine")["total_amount"].mean().sort_values(ascending=False)

cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64

In [13]:
user_spend = merged_df.groupby("user_id")["total_amount"].sum()
user_spend[user_spend > 1000].count()

np.int64(2544)

In [14]:
bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ["3.0–3.5", "3.6–4.0", "4.1–4.5", "4.6–5.0"]

merged_df["rating_range"] = pd.cut(merged_df["rating"], bins=bins, labels=labels)

merged_df.groupby("rating_range")["total_amount"].sum().sort_values(ascending=False)

C:\Users\user\AppData\Local\Temp\ipykernel_11276\3481451744.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merged_df.groupby("rating_range")["total_amount"].sum().sort_values(ascending=False)


rating_range
4.6–5.0    2197030.75
4.1–4.5    1960326.26
3.0–3.5    1881754.57
3.6–4.0    1717494.41
Name: total_amount, dtype: float64

In [15]:
merged_df[merged_df["membership"] == "Gold"] \
    .groupby("city")["total_amount"].mean().sort_values(ascending=False)

city
Chennai      808.459080
Hyderabad    806.421034
Bangalore    793.223756
Pune         781.162243
Name: total_amount, dtype: float64

In [16]:
merged_df.groupby("cuisine").agg(
    restaurants=("restaurant_id", "nunique"),
    revenue=("total_amount", "sum")
).sort_values("restaurants")

,restaurants,revenue
cuisine,,
Chinese,120,1930504.65
Indian,126,1971412.58
Italian,126,2024203.80
Mexican,128,2085503.09


In [17]:
gold_orders = merged_df[merged_df["membership"] == "Gold"].shape[0]
total_orders = merged_df.shape[0]

percentage = (gold_orders / total_orders) * 100
round(percentage)

50

In [18]:
restaurant_stats = merged_df.groupby("restaurant_name").agg(
    avg_order=("total_amount", "mean"),
    total_orders=("order_id", "count")
)

restaurant_stats[restaurant_stats["total_orders"] < 20] \
    .sort_values("avg_order", ascending=False)

KeyError: 'restaurant_name'

In [20]:
merged_df.groupby(["membership", "cuisine"])["total_amount"] \
    .sum().sort_values(ascending=False)

membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [21]:
merged_df.groupby("quarter")["total_amount"].sum().sort_values(ascending=False)

quarter
2023Q3    2037385.10
2023Q4    2018263.66
2023Q1    1993425.14
2023Q2    1945348.72
2024Q1      17201.50
Freq: Q-DEC, Name: total_amount, dtype: float64

NameError: name 'gold_df' is not defined